# Qwen2.5-0.5B-Instruct QLoRA - reproducible Colab run

This notebook trains and evaluates the Qwen QLoRA model on the same frozen hotel-review test set used by the other model families. It creates real, reusable artifacts: the adapter and tokenizer, ordered test logits and labels, generated predictions, dataset fingerprints, metrics, timing, peak VRAM, and a downloadable ZIP.

Before running it:

1. In Colab choose **Runtime → Change runtime type → T4 GPU** (or a better NVIDIA GPU).
2. Run every cell in order in a fresh runtime.
3. When prompted, upload `artifacts/hotel_review_legacy_splits.zip` (recommended), or select `train.parquet`, `dev.parquet`, and `test.parquet` together. The notebook refuses a different or reordered frozen split.
4. Download the final ZIP and send back at least `metrics.json` (preferably the whole ZIP).

No output cells are pre-populated. Metrics appear only after a real run.


## 1. Experiment settings

The default `TRAIN_CAP_TOTAL=20_000` matches the repository's QLoRA configuration and draws 10,000 examples from each class. Set it to `None` to train on all 118,990 frozen training rows. The selected subset and the full source splits are fingerprinted in the result.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/krimits/hotel-review-nlp.git"
BRANCH = "main"
REPO_DIR = Path("/content/hotel-review-nlp")

DATA_MODE = "upload"  # "upload" or "drive"
DRIVE_PROCESSED_DIR = "/content/drive/MyDrive/hotel-review-nlp/data/processed"

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
SEED = 42
TRAIN_CAP_TOTAL = 20_000  # set to None for all 118,990 training rows
DEV_CAP_TOTAL = 2_000     # set to None for the full frozen dev split
MAX_LENGTH = 320
EPOCHS = 1
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2.0e-4
MAX_NEW_TOKENS = 4

RUN_NAME = "qwen_qlora_frozen_v1"
ENABLE_TRACKIO = True
TRACKIO_PROJECT = "hotel-review-nlp"
TRACKIO_SPACE_ID = None  # local Colab logging; no HF token or Space required
RUN_DIR = REPO_DIR / "runs" / RUN_NAME
PROCESSED_DIR = REPO_DIR / "data" / "processed"


## 2. Clone the flattened repository and install a pinned Colab stack

The layout assertion catches the former nested-repository structure. This notebook deliberately uses `transformers.Trainer` with explicit completion masking, avoiding TRL API drift across Colab images. Both the repository and this notebook use the valid `BitsAndBytesConfig` argument `bnb_4bit_use_double_quant`.


In [ ]:
import os
import subprocess
import sys

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print(f"Reusing existing checkout: {REPO_DIR}")

os.chdir(REPO_DIR)
assert Path("pyproject.toml").is_file(), "pyproject.toml is missing from the repository root"
assert not Path("hotel-review-nlp/pyproject.toml").exists(), (
    "Nested project detected. Push the flattened repository before running this notebook."
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements-colab.txt",
        "datasets==4.0.0",
        "bitsandbytes==0.47.0",
        "trackio==0.35.0",
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("Repository commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## 3. CUDA and package preflight

In [ ]:
import importlib.metadata as metadata
import inspect
import platform

import torch
from transformers import BitsAndBytesConfig

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "CUDA is unavailable. Select a GPU runtime and restart."
assert "bnb_4bit_use_double_quant" in inspect.signature(BitsAndBytesConfig.__init__).parameters

gpu = torch.cuda.get_device_properties(0)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
PACKAGE_NAMES = [
    "torch", "transformers", "peft", "accelerate", "bitsandbytes", "datasets", "trackio"
]
environment = {
    "python": platform.python_version(),
    "packages": {name: metadata.version(name) for name in PACKAGE_NAMES},
    "cuda_runtime": torch.version.cuda,
    "gpu_name": gpu.name,
    "gpu_compute_capability": f"{gpu.major}.{gpu.minor}",
    "gpu_total_vram_gib": round(gpu.total_memory / 1024**3, 3),
    "compute_dtype": str(compute_dtype).replace("torch.", ""),
}
print(environment)


## 4. Supply the three frozen parquet files

`data/processed/` is intentionally ignored by Git, so a fresh clone does not contain these files. With the default `DATA_MODE="upload"`, upload `artifacts/hotel_review_legacy_splits.zip` from your local checkout, or select all three parquet files together. For Drive mode, set `DRIVE_PROCESSED_DIR` above and change `DATA_MODE` to `"drive"`.


In [ ]:
import io
import shutil
import zipfile

SPLIT_FILENAMES = ["train.parquet", "dev.parquet", "test.parquet"]
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

missing = [name for name in SPLIT_FILENAMES if not (PROCESSED_DIR / name).is_file()]
if missing and DATA_MODE == "upload":
    from google.colab import files

    print("Upload hotel_review_legacy_splits.zip, or select all three parquet files")
    uploaded = files.upload()
    by_basename = {Path(name).name: payload for name, payload in uploaded.items()}
    uploaded_zips = [name for name in uploaded if Path(name).suffix.casefold() == ".zip"]
    if uploaded_zips:
        if len(uploaded_zips) != 1:
            raise ValueError("Upload exactly one data ZIP")
        with zipfile.ZipFile(io.BytesIO(uploaded[uploaded_zips[0]])) as archive:
            members_by_basename = {Path(name).name: name for name in archive.namelist()}
            for name in SPLIT_FILENAMES:
                if name not in members_by_basename:
                    raise FileNotFoundError(f"ZIP does not include {name}")
                (PROCESSED_DIR / name).write_bytes(archive.read(members_by_basename[name]))
            manifest_name = "legacy_dataset_manifest.json"
            if manifest_name in members_by_basename:
                (PROCESSED_DIR / manifest_name).write_bytes(
                    archive.read(members_by_basename[manifest_name])
                )
    else:
        for name in SPLIT_FILENAMES:
            if name not in by_basename:
                raise FileNotFoundError(f"Upload did not include {name}")
            (PROCESSED_DIR / name).write_bytes(by_basename[name])
elif missing and DATA_MODE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    source_dir = Path(DRIVE_PROCESSED_DIR)
    for name in SPLIT_FILENAMES:
        source = source_dir / name
        if not source.is_file():
            raise FileNotFoundError(source)
        shutil.copy2(source, PROCESSED_DIR / name)
elif missing:
    raise ValueError('DATA_MODE must be either "upload" or "drive"')

for name in SPLIT_FILENAMES:
    path = PROCESSED_DIR / name
    assert path.is_file(), f"Missing {path}"
    print(name, f"{path.stat().st_size:,} bytes")


## 5. Verify the exact ordered frozen splits

The SHA-256 below hashes length-prefixed UTF-8 `text,label` pairs in row order. This is independent of parquet serialization. Training stops if any row, label, text, or ordering differs from the audited frozen dataset.


In [ ]:
import hashlib
import json

import pandas as pd

EXPECTED_FINGERPRINTS = {
    "train": {
        "rows": 118990,
        "negative": 26232,
        "positive": 92758,
        "sha256": "1a5111176aafc286a7c784c9dec3abc2528b4b94d67fc8e7000ba4df4bfbd627",
    },
    "dev": {
        "rows": 14872,
        "negative": 3278,
        "positive": 11594,
        "sha256": "7c080936decc0b470cec4eb162c121fc1245643ba095fa4d602529bc9baa8e9f",
    },
    "test": {
        "rows": 13278,
        "negative": 3278,
        "positive": 10000,
        "sha256": "a02c21271639640d645d729b959ce666671d4ba4ed5abfd43e6f9b5729245730",
    },
}
LABEL_NAMES = ("negative", "positive")


def ordered_frame_fingerprint(frame: pd.DataFrame) -> dict:
    required = {"text", "label"}
    if not required.issubset(frame.columns):
        raise ValueError(f"Missing required columns: {sorted(required - set(frame.columns))}")
    if frame[["text", "label"]].isna().any().any():
        raise ValueError("text and label must not contain nulls")
    if frame["text"].astype(str).str.strip().eq("").any():
        raise ValueError("Empty reviews are not allowed")
    observed = set(frame["label"].astype(str).unique())
    if observed != set(LABEL_NAMES):
        raise ValueError(f"Unexpected labels: {sorted(observed)}")

    digest = hashlib.sha256()
    for text, label in frame[["text", "label"]].itertuples(index=False, name=None):
        for value in (str(text), str(label)):
            encoded = value.encode("utf-8")
            digest.update(len(encoded).to_bytes(8, byteorder="big"))
            digest.update(encoded)
    counts = frame["label"].value_counts().to_dict()
    return {
        "rows": int(len(frame)),
        "negative": int(counts.get("negative", 0)),
        "positive": int(counts.get("positive", 0)),
        "sha256": digest.hexdigest(),
    }


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


frames = {
    split: pd.read_parquet(PROCESSED_DIR / f"{split}.parquet").reset_index(drop=True)
    for split in ("train", "dev", "test")
}
source_fingerprints = {split: ordered_frame_fingerprint(frame) for split, frame in frames.items()}
if source_fingerprints != EXPECTED_FINGERPRINTS:
    print(json.dumps(source_fingerprints, indent=2))
    raise RuntimeError("Frozen split mismatch: do not train or compare metrics from these files")

parquet_file_hashes = {
    split: file_sha256(PROCESSED_DIR / f"{split}.parquet") for split in frames
}
print("Frozen split verification passed")
print(json.dumps(source_fingerprints, indent=2))


## 6. Select and fingerprint the configured train/dev rows

In [ ]:
import numpy as np


def stratified_cap(frame: pd.DataFrame, cap: int | None, seed: int) -> pd.DataFrame:
    if cap is None:
        return frame.copy().reset_index(drop=True)
    if cap < 2 or cap % 2:
        raise ValueError("A finite cap must be an even integer of at least 2")
    per_class = cap // 2
    selected_indices = []
    for offset, label in enumerate(LABEL_NAMES):
        candidates = frame.index[frame["label"] == label].to_series()
        if len(candidates) < per_class:
            raise ValueError(f"Not enough {label} rows for cap={cap}")
        selected_indices.extend(
            candidates.sample(n=per_class, random_state=seed + offset, replace=False).tolist()
        )
    return frame.loc[sorted(selected_indices)].reset_index(drop=True)


train_frame = stratified_cap(frames["train"], TRAIN_CAP_TOTAL, SEED)
dev_frame = stratified_cap(frames["dev"], DEV_CAP_TOTAL, SEED + 10)
test_frame = frames["test"].copy()

selected_fingerprints = {
    "train": ordered_frame_fingerprint(train_frame),
    "dev": ordered_frame_fingerprint(dev_frame),
    "test": ordered_frame_fingerprint(test_frame),
}
assert selected_fingerprints["test"] == EXPECTED_FINGERPRINTS["test"]
print(json.dumps(selected_fingerprints, indent=2))


## 7. Load the 4-bit base and attach LoRA

NF4 with double quantization is used for the frozen base. Only the LoRA adapters are trainable.


In [ ]:
import random

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    device_map={"": 0},
    torch_dtype=compute_dtype,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_parameters = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"
    ],
}
model = get_peft_model(
    model,
    LoraConfig(
        **lora_parameters,
        bias="none",
        task_type="CAUSAL_LM",
    ),
)
model.print_trainable_parameters()
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
total_parameters = sum(parameter.numel() for parameter in model.parameters())
print({"trainable": trainable_parameters, "total": total_parameters})


## 8. Tokenize with completion-only loss

Prompt tokens and padding receive label `-100`; loss is computed only on the one-word label and `<|im_end|>`. Review text is shortened as needed while preserving the system prompt and assistant prefix.


In [ ]:
from datasets import Dataset

from reviewnlp.llm.prompt_format import format_prompt

completion_ids = {
    label: tokenizer.encode(f"{label}<|im_end|>\n", add_special_tokens=False)
    for label in LABEL_NAMES
}
prompt_token_limit = MAX_LENGTH - max(len(ids) for ids in completion_ids.values())
empty_prompt_length = len(tokenizer.encode(format_prompt(""), add_special_tokens=False))


def prompt_ids_for_review(review: str) -> list[int]:
    raw_review_ids = tokenizer.encode(" ".join(str(review).split()), add_special_tokens=False)
    keep = min(len(raw_review_ids), max(1, prompt_token_limit - empty_prompt_length - 4))
    while True:
        shortened_review = tokenizer.decode(raw_review_ids[:keep], skip_special_tokens=True)
        prompt_ids = tokenizer.encode(format_prompt(shortened_review), add_special_tokens=False)
        if len(prompt_ids) <= prompt_token_limit:
            return prompt_ids
        overflow = len(prompt_ids) - prompt_token_limit
        next_keep = max(0, keep - max(1, overflow))
        if next_keep == keep:
            raise RuntimeError("Could not fit prompt into MAX_LENGTH")
        keep = next_keep


def encode_training_example(example: dict) -> dict:
    prompt_ids = prompt_ids_for_review(example["text"])
    target_ids = completion_ids[example["label"]]
    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids.copy()
    padding = MAX_LENGTH - len(input_ids)
    if padding < 0:
        raise RuntimeError("Encoded example exceeds MAX_LENGTH")
    return {
        "input_ids": input_ids + [tokenizer.pad_token_id] * padding,
        "attention_mask": [1] * len(input_ids) + [0] * padding,
        "labels": labels + [-100] * padding,
    }


def encode_frame(frame: pd.DataFrame, description: str) -> Dataset:
    dataset = Dataset.from_pandas(frame[["text", "label"]], preserve_index=False)
    return dataset.map(
        encode_training_example,
        remove_columns=dataset.column_names,
        desc=description,
    )


train_dataset = encode_frame(train_frame, "Tokenizing training rows")
dev_dataset = encode_frame(dev_frame, "Tokenizing development rows")
first = train_dataset[0]
assert sum(value != -100 for value in first["labels"]) > 0
assert all(
    label == -100
    for label, mask in zip(first["labels"], first["attention_mask"], strict=True)
    if mask == 0
)
print(train_dataset, dev_dataset)


## 9. Train and save the adapter/tokenizer

In [ ]:
import time

from transformers import Trainer, TrainingArguments, default_data_collator

RUN_DIR.mkdir(parents=True, exist_ok=True)
os.environ["TRACKIO_PROJECT"] = TRACKIO_PROJECT
if TRACKIO_SPACE_ID is None:
    os.environ.pop("TRACKIO_SPACE_ID", None)
else:
    os.environ["TRACKIO_SPACE_ID"] = TRACKIO_SPACE_ID
training_args = TrainingArguments(
    output_dir=str(RUN_DIR / "trainer"),
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="paged_adamw_8bit",
    fp16=compute_dtype == torch.float16,
    bf16=compute_dtype == torch.bfloat16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="no",
    report_to=["trackio"] if ENABLE_TRACKIO else [],
    run_name=RUN_NAME,
    seed=SEED,
    data_seed=SEED,
    remove_unused_columns=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=default_data_collator,
)

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
training_started = time.perf_counter()
train_result = trainer.train()
if ENABLE_TRACKIO:
    import trackio

    trackio.finish()
torch.cuda.synchronize()
training_seconds = time.perf_counter() - training_started
training_peak_vram_bytes = torch.cuda.max_memory_allocated()

adapter_dir = RUN_DIR / "adapter"
model.save_pretrained(adapter_dir, safe_serialization=True)
tokenizer.save_pretrained(adapter_dir)
trainer.state.save_to_json(str(RUN_DIR / "trainer_state.json"))

lora_metadata = {
    "model": BASE_MODEL,
    "labels": list(LABEL_NAMES),
    "lora": lora_parameters,
    "quantization": {
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_use_double_quant": True,
        "bnb_4bit_compute_dtype": environment["compute_dtype"],
    },
}
(adapter_dir / "lora_config_used.json").write_text(
    json.dumps(lora_metadata, indent=2), encoding="utf-8"
)
print(f"Training time: {training_seconds / 60:.2f} minutes")
print(f"Training peak allocated VRAM: {training_peak_vram_bytes / 1024**3:.3f} GiB")
print(f"Adapter saved to {adapter_dir}")


## 10. Evaluate every frozen test row by deterministic generation

The model generates freely with greedy decoding. Parsing is strict: only exactly `negative` or `positive` (after whitespace and special-token removal) is valid. Invalid generations are counted and scored as incorrect. `test_logits.npy` stores the real first-generation-step logits for the canonical label tokens, ordered as `[negative, positive]`.


In [ ]:
import re

from tqdm.auto import tqdm

model.eval()
model.config.use_cache = True
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
tokenizer.padding_side = "left"

label_first_token_ids = {
    label: tokenizer.encode(label, add_special_tokens=False) for label in LABEL_NAMES
}
if any(len(ids) != 1 for ids in label_first_token_ids.values()):
    raise RuntimeError(
        f"Canonical labels are not single tokens for this tokenizer: {label_first_token_ids}"
    )
ordered_label_token_ids = [label_first_token_ids[label][0] for label in LABEL_NAMES]


def strict_generated_label(text: str) -> str | None:
    normalized = re.sub(r"\s+", " ", str(text)).strip().casefold()
    return normalized if normalized in LABEL_NAMES else None


generated_predictions = []
raw_generations = []
test_logits_parts = []
test_texts = test_frame["text"].astype(str).tolist()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
evaluation_started = time.perf_counter()

for start in tqdm(range(0, len(test_texts), EVAL_BATCH_SIZE), desc="Generating test labels"):
    texts = test_texts[start : start + EVAL_BATCH_SIZE]
    encoded_prompts = [prompt_ids_for_review(text) for text in texts]
    batch = tokenizer.pad(
        [
            {"input_ids": ids, "attention_mask": [1] * len(ids)}
            for ids in encoded_prompts
        ],
        padding=True,
        return_tensors="pt",
    )
    batch = {name: tensor.to(model.device) for name, tensor in batch.items()}
    with torch.inference_mode():
        generated = model.generate(
            **batch,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            return_dict_in_generate=True,
            output_scores=True,
        )
    new_token_ids = generated.sequences[:, batch["input_ids"].shape[1] :]
    decoded = tokenizer.batch_decode(new_token_ids, skip_special_tokens=True)
    parsed = [strict_generated_label(text) for text in decoded]
    generated_predictions.extend(label if label is not None else "invalid" for label in parsed)
    raw_generations.extend(decoded)
    first_step_logits = generated.scores[0][:, ordered_label_token_ids]
    test_logits_parts.append(first_step_logits.float().cpu().numpy())

torch.cuda.synchronize()
evaluation_seconds = time.perf_counter() - evaluation_started
evaluation_peak_vram_bytes = torch.cuda.max_memory_allocated()
test_logits = np.concatenate(test_logits_parts, axis=0).astype(np.float32, copy=False)
generated_predictions = np.asarray(generated_predictions)
gold_labels = test_frame["label"].astype(str).to_numpy()
numeric_gold_labels = np.asarray([0 if label == "negative" else 1 for label in gold_labels], dtype=np.int64)

assert len(generated_predictions) == len(test_frame) == test_logits.shape[0]
assert test_logits.shape == (len(test_frame), 2)
print(f"Evaluation time: {evaluation_seconds / 60:.2f} minutes")
print(f"Evaluation peak allocated VRAM: {evaluation_peak_vram_bytes / 1024**3:.3f} GiB")
print("Invalid generations:", int(np.sum(generated_predictions == "invalid")))


## 11. Compute metrics and write ordered evaluation artifacts

In [ ]:
from datetime import datetime, timezone

from sklearn.metrics import precision_recall_fscore_support


def classification_metrics(gold: np.ndarray, predicted: np.ndarray) -> dict:
    precision, recall, f1, support = precision_recall_fscore_support(
        gold, predicted, labels=list(LABEL_NAMES), zero_division=0
    )
    macro = precision_recall_fscore_support(
        gold, predicted, labels=list(LABEL_NAMES), average="macro", zero_division=0
    )
    weighted_f1 = precision_recall_fscore_support(
        gold, predicted, labels=list(LABEL_NAMES), average="weighted", zero_division=0
    )[2]
    columns = ["negative", "positive", "invalid"]
    matrix = [
        [int(np.sum((gold == actual) & (predicted == value))) for value in columns]
        for actual in LABEL_NAMES
    ]
    return {
        "accuracy": round(float(np.mean(gold == predicted)), 4),
        "macro_precision": round(float(macro[0]), 4),
        "macro_recall": round(float(macro[1]), 4),
        "macro_f1": round(float(macro[2]), 4),
        "weighted_f1": round(float(weighted_f1), 4),
        "per_class": {
            label: {
                "precision": round(float(precision[index]), 4),
                "recall": round(float(recall[index]), 4),
                "f1": round(float(f1[index]), 4),
                "support": int(support[index]),
            }
            for index, label in enumerate(LABEL_NAMES)
        },
        "confusion_matrix": matrix,
        "confusion_matrix_rows": list(LABEL_NAMES),
        "confusion_matrix_columns": columns,
    }


generation_metrics = classification_metrics(gold_labels, generated_predictions)
score_predictions = np.asarray(LABEL_NAMES)[test_logits.argmax(axis=1)]
first_step_score_metrics = classification_metrics(gold_labels, score_predictions)
invalid_mask = generated_predictions == "invalid"
invalid_samples = [
    {
        "test_row": int(index),
        "gold": str(gold_labels[index]),
        "raw_generation": str(raw_generations[index]),
    }
    for index in np.flatnonzero(invalid_mask)[:25]
]

test_row_ids = np.asarray(
    [hashlib.sha256(str(text).encode("utf-8")).hexdigest() for text in test_texts]
)
np.save(RUN_DIR / "test_logits.npy", test_logits)
np.save(RUN_DIR / "test_labels.npy", numeric_gold_labels)
np.save(RUN_DIR / "test_generated_predictions.npy", generated_predictions)
np.save(RUN_DIR / "test_row_ids.npy", test_row_ids)

with (RUN_DIR / "raw_generations.jsonl").open("w", encoding="utf-8") as handle:
    for index, (gold, predicted, raw) in enumerate(
        zip(gold_labels, generated_predictions, raw_generations, strict=True)
    ):
        handle.write(
            json.dumps(
                {
                    "test_row": index,
                    "gold": str(gold),
                    "parsed_prediction": str(predicted),
                    "raw_generation": str(raw),
                },
                ensure_ascii=False,
            )
            + "\n"
        )

git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
train_metrics = {
    key: float(value) if isinstance(value, (float, np.floating)) else int(value)
    if isinstance(value, (int, np.integer))
    else value
    for key, value in train_result.metrics.items()
}
metrics = {
    "schema_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_family": "Qwen2.5-0.5B-Instruct QLoRA",
    "base_model": BASE_MODEL,
    "repository": {"url": REPO_URL, "branch": BRANCH, "commit": git_commit},
    "environment": environment,
    "dataset": {
        "identity": "audited frozen v1 ordered splits",
        "source_fingerprints": source_fingerprints,
        "selected_fingerprints": selected_fingerprints,
        "parquet_file_sha256": parquet_file_hashes,
        "test_row_id_policy": "sha256 of exact UTF-8 review text, in frozen test order",
    },
    "training": {
        "train_cap_total": TRAIN_CAP_TOTAL,
        "dev_cap_total": DEV_CAP_TOTAL,
        "epochs": EPOCHS,
        "max_length": MAX_LENGTH,
        "batch_size": TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch_size": TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "seed": SEED,
        "completion_only_loss": True,
        "trainable_parameters": trainable_parameters,
        "total_parameters_visible": total_parameters,
        "lora": lora_parameters,
        "seconds": round(training_seconds, 3),
        "peak_allocated_vram_bytes": int(training_peak_vram_bytes),
        "trainer_metrics": train_metrics,
    },
    "evaluation": {
        "primary_prediction_method": "greedy generated label",
        "generation_metrics": generation_metrics,
        "invalid_output_count": int(invalid_mask.sum()),
        "invalid_output_rate": round(float(invalid_mask.mean()), 8),
        "invalid_output_samples": invalid_samples,
        "seconds": round(evaluation_seconds, 3),
        "rows_per_second": round(len(test_frame) / evaluation_seconds, 3),
        "peak_allocated_vram_bytes": int(evaluation_peak_vram_bytes),
        "max_new_tokens": MAX_NEW_TOKENS,
        "test_logits_semantics": (
            "raw first-generation-step logits for canonical label tokens in "
            "[negative, positive] order"
        ),
        "first_step_score_metrics": first_step_score_metrics,
    },
    "artifacts": {
        "test_logits": "test_logits.npy",
        "test_labels": "test_labels.npy",
        "generated_predictions": "test_generated_predictions.npy",
        "test_row_ids": "test_row_ids.npy",
        "raw_generations": "raw_generations.jsonl",
        "adapter": "adapter/",
    },
}
(RUN_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
(RUN_DIR / "dataset_fingerprints.json").write_text(
    json.dumps(metrics["dataset"], indent=2), encoding="utf-8"
)
print(json.dumps(metrics["evaluation"], indent=2))


## 12. Verify, ZIP, and download the complete run

In [ ]:
required_artifacts = [
    RUN_DIR / "adapter" / "adapter_config.json",
    RUN_DIR / "adapter" / "adapter_model.safetensors",
    RUN_DIR / "adapter" / "tokenizer_config.json",
    RUN_DIR / "test_logits.npy",
    RUN_DIR / "test_labels.npy",
    RUN_DIR / "test_generated_predictions.npy",
    RUN_DIR / "test_row_ids.npy",
    RUN_DIR / "metrics.json",
    RUN_DIR / "dataset_fingerprints.json",
]
for path in required_artifacts:
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f"Missing or empty artifact: {path}")

reloaded_logits = np.load(RUN_DIR / "test_logits.npy")
reloaded_labels = np.load(RUN_DIR / "test_labels.npy")
assert reloaded_logits.shape == (EXPECTED_FINGERPRINTS["test"]["rows"], 2)
assert reloaded_labels.shape == (EXPECTED_FINGERPRINTS["test"]["rows"],)
assert json.loads((RUN_DIR / "metrics.json").read_text(encoding="utf-8"))["dataset"][
    "source_fingerprints"
]["test"] == EXPECTED_FINGERPRINTS["test"]

artifact_manifest = {
    str(path.relative_to(RUN_DIR)): {
        "bytes": path.stat().st_size,
        "sha256": file_sha256(path),
    }
    for path in sorted(RUN_DIR.rglob("*"))
    if path.is_file() and path.name != "artifact_manifest.json"
}
(RUN_DIR / "artifact_manifest.json").write_text(
    json.dumps(artifact_manifest, indent=2), encoding="utf-8"
)

zip_path = Path(
    shutil.make_archive(
        str(REPO_DIR / RUN_NAME),
        "zip",
        root_dir=RUN_DIR.parent,
        base_dir=RUN_DIR.name,
    )
)
print(f"Created {zip_path} ({zip_path.stat().st_size / 1024**2:.2f} MiB)")

COPY_TO_DRIVE = False
if COPY_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive") / zip_path.name
    shutil.copy2(zip_path, destination)
    print("Copied to", destination)

from google.colab import files

files.download(str(zip_path))


Send back `metrics.json` and, if possible, the whole downloaded ZIP. The ZIP contains enough provenance to verify the frozen test set, reproduce the result table, and load the adapter for the unified benchmark.
